# 🧪 Stonelab Challenge — Case B: Churn de Clientes
**Sistemas Inteligentes — Atividade Prática 2 (2026/1)**

Pipeline: dados tabulares de lojistas → previsão de churn (30 dias) → explicação em linguagem natural.

**Integrantes:** _(preencher)_

**Estrutura:** 4.1 Representação · 4.2 EDA · 4.3 Modelagem · 4.4 Geração de Texto (3 abordagens) · 5 Avaliação · Referências.

## 0. Setup

In [ ]:
# Instala dependências que não vêm por padrão no Colab
!pip install -q transformers rouge-score sacrebleu 2>/dev/null

import pandas as pd, numpy as np, io
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, f1_score)
import warnings; warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

### Carga dos dados
Lido direto do repositório do grupo (CSV bruto, sem edição manual) — reprodutível em qualquer ambiente.

In [ ]:
import urllib.request, csv, io

URL = "https://raw.githubusercontent.com/RitaLB/Stonelab-Churn-Clientes-AI/main/data/churn_clientes.csv"
texto = urllib.request.urlopen(URL).read().decode("utf-8")
raw = list(csv.reader(io.StringIO(texto)))  # leitura linha a linha, tolera linhas malformadas

### Diagnóstico de integridade do arquivo
O dataset bruto pode conter **linhas malformadas** (número de campos diferente do cabeçalho, por erro de exportação). Antes de qualquer análise, verificamos a integridade estrutural de forma **genérica** — a checagem não depende de índices fixos, funcionando para qualquer dataset com o mesmo cabeçalho.

In [ ]:
header = raw[0]
n_cols = len(header)
malformadas = [(i, len(r), r) for i, r in enumerate(raw[1:], start=1) if len(r) != n_cols]

print(f"Colunas esperadas (cabeçalho): {n_cols}")
print(f"Linhas de dados: {len(raw) - 1}")
print(f"Linhas malformadas: {len(malformadas)}")
for i, ncampos, r in malformadas:
    print(f"  linha {i}: {ncampos} campos -> {r}")

### Correção das linhas malformadas — justificativa

**Diagnóstico:** as linhas malformadas apresentam **um campo a mais** que o cabeçalho, com um campo **vazio** intercalado (`...,43.00,,3,...`). Isso é a assinatura de uma **vírgula espúria** inserida na exportação, não um valor legítimo faltante.

**Regra de correção (genérica e auditável):** para toda linha com `n_campos == n_cols + 1` que contenha exatamente um campo vazio, removemos esse campo vazio, restaurando o alinhamento. Isso **preserva o registro** (não descartamos a linha) e **não fabrica dados**.

**Distinção importante:** essa correção trata *erro estrutural do arquivo*. Valores genuinamente ausentes (ex.: `nps_ultimo`, `plano` vazios em linhas bem-formadas) são **mantidos como faltantes** e tratados na etapa de imputação (seção 4.1), como manda o enunciado.

In [ ]:
def corrigir_linha(r, n_cols):
    """Corrige linha com 1 campo extra causado por vírgula espúria (1 campo vazio)."""
    if len(r) == n_cols + 1 and r.count("") == 1:
        r = [c for c in r if c != ""]  # remove o único campo vazio espúrio
    return r

dados_corrigidos = [corrigir_linha(r, n_cols) for r in raw[1:]]

# Validação pós-correção: todas as linhas devem ter n_cols campos
restantes = [i for i, r in enumerate(dados_corrigidos, 1) if len(r) != n_cols]
assert not restantes, f"Linhas ainda malformadas: {restantes}"
print("Correção aplicada. Todas as linhas agora têm", n_cols, "campos.")

### Construção do DataFrame
Campos vazios legítimos são convertidos em `NaN` (tratados como faltantes na seção 4.1).

In [ ]:
df = pd.DataFrame(dados_corrigidos, columns=header).replace("", np.nan)

# Reconstrói os tipos numéricos (leitura foi feita como string para inspeção)
num_raw = ["tpv_medio_mensal","variacao_tpv_3m","dias_sem_transacao","tempo_cliente_meses",
           "num_produtos_ativos","ticket_medio","num_chamados_suporte","nps_ultimo","churn"]
for c in num_raw:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["churn"] = df["churn"].astype(int)

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 4.1 Representação do Conhecimento

Decisões de pré-processamento, com justificativa:

| Item | Decisão | Justificativa |
|---|---|---|
| `cliente_id` | **descartada** | Identificador; não carrega sinal preditivo (dicionário alerta explicitamente). |
| Numéricas | **padronização (StandardScaler)** | Escalas muito diferentes (`tpv_medio_mensal` na casa dos milhares vs. `nps_ultimo` 0–10). Padronizar evita que features de escala grande dominem a Regressão Logística. |
| Categóricas nominais (`segmento`, `regiao`, `plano`, `tem_maquininha`, `usa_app_mobile`) | **One-Hot Encoding** | Não há ordem natural entre categorias; One-Hot evita impor ordinalidade falsa (erro que `LabelEncoder` cometeria). |
| Faltantes numéricos | **imputação pela mediana** | Robusta a outliers, adequada ao dataset pequeno. Afeta `nps_ultimo`. |
| Faltantes categóricos | **imputação pela moda** | Categoria mais frequente. Afeta `plano`. |

Todo o pré-processamento é encapsulado num `ColumnTransformer` dentro de um `Pipeline`, garantindo que a padronização/imputação seja aprendida **só no treino** (sem vazamento para o teste/validação).

In [ ]:
TARGET = "churn"
DROP = ["cliente_id", TARGET]

num_cols = df.drop(columns=DROP).select_dtypes(include=np.number).columns.tolist()
cat_cols = df.drop(columns=DROP).select_dtypes(include="object").columns.tolist()

print("Numéricas :", num_cols)
print("Categóricas:", cat_cols)

preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

X = df.drop(columns=DROP)
y = df[TARGET]

## 4.2 Análise Exploratória (EDA)

### 4.2.1 Valores faltantes

In [ ]:
miss = df.isnull().sum()
print(miss[miss > 0] if miss.sum() else "Sem valores faltantes.")

### 4.2.2 Distribuição da variável-alvo (desbalanceamento)

In [ ]:
fig, ax = plt.subplots(figsize=(5,3))
df[TARGET].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#2ecc71","#e74c3c"])
ax.set_xticklabels(["Ativo (0)","Churn (1)"], rotation=0); ax.set_ylabel("Contagem")
ax.set_title("Distribuição da variável-alvo"); plt.tight_layout(); plt.show()
print(f"Proporção de churn: {df[TARGET].mean():.1%}")

> Classe minoritária = churn (~37%). Dataset pequeno (30 linhas): usaremos validação cruzada **estratificada** e priorizaremos **F1** e **AUC** sobre acurácia.

### 4.2.3 Numéricas por classe

In [ ]:
feat_num = ["variacao_tpv_3m","dias_sem_transacao","nps_ultimo","tpv_medio_mensal"]
fig, axes = plt.subplots(2,2, figsize=(10,7))
for c, ax in zip(feat_num, axes.flatten()):
    for v,color,lab in [(0,"#2ecc71","Ativo"),(1,"#e74c3c","Churn")]:
        ax.hist(df[df[TARGET]==v][c].dropna(), bins=8, alpha=.6, color=color, label=lab)
    ax.set_title(c); ax.legend(fontsize=8)
plt.suptitle("Distribuições por classe"); plt.tight_layout(); plt.show()

### 4.2.4 Correlação (numéricas)

In [ ]:
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(df[num_cols+[TARGET]].corr(numeric_only=True), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Matriz de correlação"); plt.tight_layout(); plt.show()

### 4.2.5 Categóricas vs. churn

In [ ]:
cats = ["segmento","plano","usa_app_mobile"]
fig, axes = plt.subplots(1,3, figsize=(13,4))
for c, ax in zip(cats, axes):
    pd.crosstab(df[c], df[TARGET], normalize="index").plot(
        kind="bar", stacked=True, ax=ax, color=["#2ecc71","#e74c3c"])
    ax.set_title(c); ax.set_ylabel("Proporção"); ax.legend(["Ativo","Churn"], fontsize=8)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

### 4.2.6 Hipóteses (features mais preditivas)
1. **`variacao_tpv_3m`** — queda de volume deve preceder churn (sinal comercial mais forte).
2. **`dias_sem_transacao`** — inatividade prolongada indica abandono.
3. **`nps_ultimo`** — detratores (nota baixa) tendem a sair.
4. **`num_chamados_suporte`** — muitos chamados = fricção/insatisfação.

Essas serão as variáveis privilegiadas na geração de texto por serem as mais interpretáveis para o time comercial.

## 4.3 Modelagem Preditiva

Dois modelos: **Regressão Logística** (baseline linear, interpretável) e **Random Forest** (não-linear, captura interações). Ambos dentro de um `Pipeline` com o pré-processamento.

Avaliação: **validação cruzada estratificada** (3 folds, adequado ao tamanho pequeno) com **F1** e **AUC**. Inclui **otimização de hiperparâmetros** (GridSearchCV) no Random Forest.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

def avaliar(model, nome):
    f1 = cross_val_score(model, X, y, cv=cv, scoring="f1")
    auc = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    print(f"{nome:20s} | F1 {f1.mean():.3f}±{f1.std():.3f} | AUC {auc.mean():.3f}±{auc.std():.3f}")
    return f1.mean()

logit = Pipeline([("prep", preprocess),
                  ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))])
rf = Pipeline([("prep", preprocess),
               ("clf", RandomForestClassifier(class_weight="balanced", random_state=SEED))])

avaliar(logit, "LogisticRegression")
avaliar(rf, "RandomForest")

### 4.3.1 Otimização de hiperparâmetros (Random Forest)

In [ ]:
grid = {
    "clf__n_estimators":[100,300],
    "clf__max_depth":[3,5,None],
    "clf__min_samples_leaf":[1,2],
}
gs = GridSearchCV(rf, grid, cv=cv, scoring="f1", n_jobs=-1)
gs.fit(X, y)
print("Melhores params:", gs.best_params_)
print("Melhor F1 (CV):", round(gs.best_score_,3))
best_rf = gs.best_estimator_

### 4.3.2 Matriz de confusão e relatório (holdout estratificado)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.3, stratify=y, random_state=SEED)

for model, nome in [(logit,"Logistic Regression"), (best_rf,"Random Forest (tuned)")]:
    model.fit(X_tr, y_tr)
    yp = model.predict(X_te)
    yp_prob = model.predict_proba(X_te)[:,1]
    print(f"\n===== {nome} =====")
    print(classification_report(y_te, yp, target_names=["Ativo","Churn"]))
    print("AUC:", round(roc_auc_score(y_te, yp_prob),3))
    fig, ax = plt.subplots(figsize=(4,3.5))
    ConfusionMatrixDisplay.from_predictions(y_te, yp, display_labels=["Ativo","Churn"],
                                            cmap="Blues", ax=ax)
    ax.set_title(nome); plt.tight_layout(); plt.show()

### 4.3.3 Importância das features (Random Forest)

In [ ]:
# Nomes das features após One-Hot
best_rf.fit(X, y)
ohe = best_rf.named_steps["prep"].named_transformers_["cat"].named_steps["oh"]
feat_names = num_cols + list(ohe.get_feature_names_out(cat_cols))
imp = pd.Series(best_rf.named_steps["clf"].feature_importances_, index=feat_names)
imp = imp.sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(7,4))
imp[::-1].plot(kind="barh", ax=ax, color="#3498db")
ax.set_title("Top 10 features — Random Forest"); plt.tight_layout(); plt.show()
imp

### 4.3.4 Interpretação
_(comentar após rodar)_ — comparar F1/AUC dos dois modelos, confirmar se as features mais importantes batem com as hipóteses da EDA (`variacao_tpv_3m`, `dias_sem_transacao`, `nps_ultimo`). Registrar a limitação do tamanho da amostra (30 linhas → alta variância entre folds).

O `best_rf` treinado no conjunto completo será usado como modelo de produção na geração de texto.

## 4.4 Geração de Texto (Tabular → Linguagem)

Três abordagens (item 4.4): **A) Templates**, **C) LLM via HuggingFace**, **D) Híbrida**.
Todas recebem uma linha do dataset, usam a **probabilidade prevista pelo modelo** para modular a confiança, e retornam uma explicação curta.

### Função auxiliar: features de risco a partir de uma linha
Categoriza os valores em rótulos legíveis ("baixo/moderado/alto") e coleta os sinais de risco relevantes.

In [ ]:
def prob_churn(row):
    return float(best_rf.predict_proba(pd.DataFrame([row[X.columns]]))[:,1][0])

def sinais_de_risco(row):
    s = []
    if row["variacao_tpv_3m"] <= -0.20:
        s.append(f"queda expressiva no volume transacionado ({row['variacao_tpv_3m']:.0%} em 3 meses)")
    elif row["variacao_tpv_3m"] < 0:
        s.append(f"leve queda no volume ({row['variacao_tpv_3m']:.0%} em 3 meses)")
    if row["dias_sem_transacao"] >= 15:
        s.append(f"{int(row['dias_sem_transacao'])} dias sem transacionar")
    if pd.notna(row["nps_ultimo"]) and row["nps_ultimo"] <= 6:
        s.append(f"NPS baixo ({int(row['nps_ultimo'])}), perfil detrator")
    if row["num_chamados_suporte"] >= 4:
        s.append(f"{int(row['num_chamados_suporte'])} chamados de suporte recentes")
    if row["num_produtos_ativos"] <= 1:
        s.append("uso de apenas um produto (baixo engajamento)")
    if str(row["usa_app_mobile"]).lower() == "nao":
        s.append("não utiliza o app mobile")
    return s

#### Abordagem A — Templates (baseline)
Regras sobre as features + probabilidade do modelo montam a frase.

In [ ]:
def explicar_templates(row):
    p = prob_churn(row)
    nivel = "ALTO" if p >= .6 else "MODERADO" if p >= .35 else "BAIXO"
    sinais = sinais_de_risco(row)
    txt = f"Cliente {row['cliente_id']} com risco {nivel} de churn (probabilidade {p:.0%}). "
    if sinais:
        txt += "Fatores: " + "; ".join(sinais) + "."
    else:
        txt += "Indicadores dentro do esperado; comportamento estável."
    return txt

#### Abordagem C — LLM via HuggingFace (prompt engineering)
Usamos um modelo instruído leve (`google/flan-t5-base`) que roda no Colab gratuito, **sem fine-tuning**: justificativa é que o prompt já fornece os dados estruturados e a tarefa é reescrita fluente — prompt engineering basta e evita custo de treino sobre um corpus minúsculo.

In [ ]:
from transformers import pipeline
gerador = pipeline("text2text-generation", model="google/flan-t5-base")

def explicar_llm(row):
    p = prob_churn(row)
    sinais = sinais_de_risco(row) or ["nenhum sinal relevante"]
    prompt = (
        "Escreva uma explicação curta e profissional, em português, para o time comercial de uma fintech, "
        f"sobre por que este lojista tem risco de churn. Probabilidade prevista: {p:.0%}. "
        f"Sinais observados: {'; '.join(sinais)}. "
        "Use no máximo duas frases e não invente dados."
    )
    out = gerador(prompt, max_new_tokens=80, do_sample=False)[0]["generated_text"]
    return out.strip()

#### Abordagem D — Híbrida
LLM gera a redação fluente; um **verificador determinístico** garante fidelidade: se o texto do LLM omitir o nível de risco ou vier vazio, cai para o template. Combina naturalidade (C) com garantia factual (A).

In [ ]:
def explicar_hibrido(row):
    p = prob_churn(row)
    base = explicar_templates(row)
    try:
        llm = explicar_llm(row)
    except Exception:
        return base
    # verificação: LLM precisa ser não-trivial e citar risco
    if len(llm) < 20 or ("risc" not in llm.lower() and "churn" not in llm.lower()):
        return base
    return f"{llm} (Risco previsto: {p:.0%}.)"  # sempre ancorado na prob real

### 4.4.1 Exemplos para 10 clientes distintos (as três abordagens)

In [ ]:
amostra = df.sample(10, random_state=SEED)
for _, row in amostra.iterrows():
    print("="*80)
    print(f"{row['cliente_id']} | churn real = {row['churn']} | "
          f"var_tpv={row['variacao_tpv_3m']:.0%}, dias_sem_transacao={row['dias_sem_transacao']}, "
          f"nps={row['nps_ultimo']}, produtos={row['num_produtos_ativos']}")
    print("[Templates]", explicar_templates(row))
    print("[LLM]      ", explicar_llm(row))
    print("[Híbrido]  ", explicar_hibrido(row))

## 5. Avaliação da Qualidade do Texto

Duas formas: **rubrica humana** (coerência, completude, fidelidade) e **métrica automática ROUGE-L** contra textos de referência.

### 5.1 Rubrica humana
Cada texto é avaliado de 1 a 3 em três critérios. Preencha `notas` manualmente após ler as saídas acima.

| Critério | 1 | 2 | 3 |
|---|---|---|---|
| Coerência | confuso | aceitável | claro e fluente |
| Completude | ignora sinais-chave | cita alguns | cita os principais |
| Fidelidade | inventa/erra dados | pequenos deslizes | 100% fiel aos dados |

In [ ]:
# Exemplo de registro da avaliação humana (ajuste as notas conforme sua leitura)
avaliacao = pd.DataFrame([
    # cliente, abordagem, coerencia, completude, fidelidade
    ("L002","Templates",3,3,3),
    ("L002","LLM",3,2,2),
    ("L002","Híbrido",3,3,3),
], columns=["cliente","abordagem","coerencia","completude","fidelidade"])
avaliacao["media"] = avaliacao[["coerencia","completude","fidelidade"]].mean(axis=1)
avaliacao.groupby("abordagem")["media"].mean()

### 5.2 Métrica automática — ROUGE-L
Comparamos a saída contra uma referência escrita à mão para alguns clientes. ROUGE-L mede sobreposição de subsequência (fluidez/cobertura).

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# Referências manuais (escreva 3-5 baseadas nos dados reais)
referencias = {
 "L002":"Cliente L002 com alto risco de churn: forte queda no volume, muitos dias sem transacionar, NPS detrator e vários chamados de suporte.",
}
for cid, ref in referencias.items():
    row = df[df.cliente_id==cid].iloc[0]
    for nome, fn in [("Templates",explicar_templates),("LLM",explicar_llm),("Híbrido",explicar_hibrido)]:
        sc = scorer.score(ref, fn(row))["rougeL"].fmeasure
        print(f"{cid} | {nome:9s} | ROUGE-L F1 = {sc:.3f}")

### 5.3 Análise crítica
_(comentar após rodar)_ — Templates: máxima fidelidade, texto mecânico. LLM: mais natural, risco de omitir/alucinar dados. Híbrido: melhor equilíbrio (fluência com âncora factual). Apontar exemplos bons e ruins observados.

## Referências e ferramentas de IA generativa

- **scikit-learn** — modelagem, pré-processamento, métricas.
- **Hugging Face Transformers** — `google/flan-t5-base` para geração de texto (Abordagem C/D).
- **rouge-score / sacrebleu** — avaliação automática de texto.
- _(Se usaram ChatGPT/Claude/etc. no desenvolvimento, declarar aqui a ferramenta e a finalidade — ex.: "Claude, para estruturar o notebook e revisar código".)_